# AXI test 2: drift over 30 steps, and what a growing chat costs
Model: **Qwen2.5-7B-Instruct in 4-bit** (same as test 1). Three runs:
* **Run A** (about 15 min): 12 episodes of 30 requests in a row. Three ways to hold the state: *blind* (apply whatever the model wrote), *gate* (refuse bad changes), *repair_m6* (gate + repair). How far does the held state drift from the right state?
* **Run B** (about 10-20 min): 4 of those episodes again, but the whole conversation stays in the model's context. Is it slower? Does the model write different things?
* **Run C** (optional, about 30 min): Run A again with the rule stated in the prompt.

**Before you start:** Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save. Run each cell **one at a time, top to bottom**, wait for the green tick before the next one.

If Colab disconnects: reconnect, run cells 1 to 4 again, then run the run-cell again. It continues where it stopped.

Predictions and reading rules were written before this run in `docs/LONG_HORIZON_TEST.md`.

### Cell 1: check the GPU (it must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-long.zip`

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the 4-bit tools (about 1 minute)

In [ ]:
!pip -q install bitsandbytes accelerate

### Cell 4: load the model (downloads about 15 GB once; about 5 minutes)

In [ ]:
import sys, json, time, os, torch
sys.path.insert(0, "/content/axi/python")
from colab.colab_llm_experiment import build_hf_llm
from axi.experiments.long_horizon import (run_long, summarize_long, compare_context, verdict_state, verdict_context,
                                          print_long, print_context, print_verdict, POLICIES)

MODEL = "Qwen/Qwen2.5-7B-Instruct"
llm, model = build_hf_llm(MODEL, "cuda", load_4bit=True)
os.makedirs("/content/out", exist_ok=True)
T0 = time.time()
def prog(i, n): print(f"  episode {i}/{n}   {time.time() - T0:6.0f} s", end="\r")
def save(res, path):
    json.dump(res, open(path, "w"), indent=1); return path
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1))

### Cell 5: quick check, 1 episode of 4 steps in both modes (about 2 minutes). If it fails, stop and send me the red error text

In [ ]:
a = run_long(llm, "state", 1, 4, 99, False, "/content/smoke_state.jsonl", meta=dict(model=MODEL, smoke=True))
print_long(a["summary"], "smoke, state mode")
b = run_long(llm, "history", 1, 4, 99, False, "/content/smoke_history.jsonl", meta=dict(model=MODEL, smoke=True))
for s in b["rows"][0]["policies"]["repair_m6"]:
    print("history step", s["t"], s["kind"], "| prompt tokens", s["prompt_tokens"], "| seconds", round(s["seconds"], 2), "|", repr(s["text"])[:70])

### Cell 6: RUN A, no rule. 12 episodes x 30 steps, 3 policies (about 15 minutes; counter shows episode i/12)

In [ ]:
T0 = time.time()
res_state = run_long(llm, "state", 12, 30, 3, False, "/content/out/axi_long_state_norule.jsonl", meta=dict(model=MODEL), progress=prog)
print()
print_long(res_state["summary"], "RUN A: state mode, no rule")
print_verdict(verdict_state(res_state["summary"]), "RUN A")
files.download(save(res_state, "/content/out/axi_long_state_norule.json"))

### Cell 7: RUN B, chat history kept. 4 episodes x 30 steps (about 10-20 minutes; the later steps are slower, that is the point)

In [ ]:
T0 = time.time()
res_hist = run_long(llm, "history", 4, 30, 3, False, "/content/out/axi_long_history_norule.jsonl", meta=dict(model=MODEL), progress=prog)
print()
print_long(res_hist["summary"], "RUN B: history mode, no rule")
print("episodes stopped early (context / oom / None):", [r["stopped"]["repair_m6"] for r in res_hist["rows"]])
cmp = compare_context(res_state["rows"], res_hist["rows"])
print_context(cmp)
print_verdict(verdict_context(cmp), "RUN B")
files.download(save(res_hist, "/content/out/axi_long_history_norule.json"))
torch.cuda.empty_cache()

### Cell 8 (optional): RUN C, rule stated in the prompt. 8 episodes x 30 steps (about 30 minutes)

In [ ]:
T0 = time.time()
res_rule = run_long(llm, "state", 8, 30, 3, True, "/content/out/axi_long_state_rule.jsonl", meta=dict(model=MODEL), progress=prog)
print()
print_long(res_rule["summary"], "RUN C: state mode, rule stated")
print_verdict(verdict_state(res_rule["summary"]), "RUN C")
files.download(save(res_rule, "/content/out/axi_long_state_rule.json"))

### Done
Files downloaded: `axi_long_state_norule.json`, `axi_long_history_norule.json` (and `axi_long_state_rule.json` if you ran Cell 8). Upload them to the chat, with a screenshot of the printed tables. Do not paste any keys anywhere.